In [1]:
# import libraries
import pandas as pd
import json
import re
import sys

# import custom helper functions
sys.path.append("../../utils/")
from preprocessing import create_regex_pattern
from ner_classification import text_to_bio, find_dictionary_matches
from custom_evaluation import extract_spans, evaluate_seqeval, mention_level_evaluation, sentence_level_evaluation

# import the annotations
with open("../../01_data/annotations_reduced.json", "r") as f:
    data = json.load(f)

# import the dictionary
group_dictionary_df = pd.read_csv("../../01_data/groups_dictionary.csv")

In [2]:
# add bio tags to the dataset
for task in data:
    bio_tags = text_to_bio(task)
    task["bio_tags"] = bio_tags

# create the regex pattern
combined_regex = create_regex_pattern(group_dictionary_df)

In [14]:
# store all bio tags in a list
gt_bio = [sent["bio_tags"] for sent in data]

# empty list to store predicted tags
pred_bio = []

# loop through tasks, find dictionary matches and append bio tags to list
for task in data:
    sentence = task["sentence"]
    pred_tags = find_dictionary_matches(sentence, combined_regex)
    pred_bio.append(pred_tags)

# evaluate at the entity level with seqeval
seqeval_metrics = evaluate_seqeval(gt_bio, pred_bio)


# evaluate on the entity level with custom cross-span metric
all_true_spans = []
all_predicted_spans = []

# get spans of all true positives and predictions
for idx in range(len(gt_bio)):
    all_true_spans.append(extract_spans(gt_bio[idx]))
    all_predicted_spans.append(extract_spans(pred_bio[idx]))
 
# apply cross-span evaluation
cross_span_metrics = mention_level_evaluation(all_true_spans, all_predicted_spans)

# evaluate at the sentence level
sentence_level_metrics = sentence_level_evaluation(gt_bio, pred_bio)

# store all metrics in a dictionary and export it
test_metrics = {}
test_metrics["dictionary_baseline"] = {
    "seqeval": seqeval_metrics,
    "cross_span": cross_span_metrics,
    "sentence_level": sentence_level_metrics
    }
with open("evaluation_metrics_dictionary.json", "w") as f:
    json.dump(test_metrics, f, indent=4)